# Electric Charge from Wave Phase

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gpartin/LFMPublicExperiments/blob/main/LFMPublicExperiments/notebooks/LFM_Charge_From_Phase.ipynb)

## What This Notebook Demonstrates

In LFM, electric charge is the **phase** of the complex wave field:
- $\theta = 0$ $\rightarrow$ electron
- $\theta = \pi$ $\rightarrow$ positron

We evolve the full complex GOV-01 + GOV-02 system with two wave packets:
- **Same phase** ($\Delta\theta = 0$): constructive interference $\rightarrow$ more energy $\rightarrow$ **repulsion**
- **Opposite phase** ($\Delta\theta = \pi$): destructive interference $\rightarrow$ less energy $\rightarrow$ **attraction**

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

CHI_0 = 19.0
KAPPA = 1.0 / 63.0
C = 1.0

N = 512
dx = 1.0
dt = 0.05  # must satisfy CFL: dt < 2/sqrt(4c^2/dx^2 + chi^2)
x = np.arange(N) * dx
n_steps = 500

def laplacian_1d(f, dx):
    """1D discrete Laplacian with zero boundaries."""
    lap = np.zeros_like(f)
    lap[1:-1] = (f[2:] + f[:-2] - 2*f[1:-1]) / dx**2
    return lap

def gaussian_complex(x, center, sigma, phase, amp=0.6):
    """Complex Gaussian wave packet with phase theta."""
    envelope = np.exp(-(x - center)**2 / (2 * sigma**2))
    return amp * envelope * np.exp(1j * phase)

def evolve_complex(psi_init, chi_init, n_steps, dt, dx, c, kappa):
    """Coupled leapfrog for complex Psi + real chi."""
    psi = psi_init.copy()
    chi = chi_init.copy()
    psi_prev = psi.copy()
    chi_prev = chi.copy()
    psi_hist = np.zeros((n_steps, len(psi)), dtype=np.complex128)
    chi_hist = np.zeros((n_steps, len(chi)))
    for t in range(n_steps):
        psi_hist[t] = psi
        chi_hist[t] = chi
        # GOV-01 (complex)
        psi_next = 2*psi - psi_prev + dt**2 * (c**2 * laplacian_1d(psi, dx) - chi**2 * psi)
        # GOV-02 (uses |Psi|^2)
        chi_next = 2*chi - chi_prev + dt**2 * (c**2 * laplacian_1d(chi, dx) - kappa * np.abs(psi)**2)
        chi_next = np.maximum(chi_next, 0.1)
        psi_prev, psi = psi, psi_next
        chi_prev, chi = chi, chi_next
    return psi_hist, chi_hist

# Configuration 1: OPPOSITE PHASES (should attract)
print("Running opposite phases (theta=0, theta=pi)...")
psi_opp = (gaussian_complex(x, 230, 25, 0)
           + gaussian_complex(x, 282, 25, np.pi))
chi_init = np.full(N, CHI_0)
psi_hist_opp, chi_hist_opp = evolve_complex(psi_opp, chi_init.copy(),
                                             n_steps, dt, dx, C, KAPPA)

# Configuration 2: SAME PHASE (should repel)
print("Running same phases (theta=0, theta=0)...")
psi_same = (gaussian_complex(x, 230, 25, 0)
            + gaussian_complex(x, 282, 25, 0))
psi_hist_same, chi_hist_same = evolve_complex(psi_same, chi_init.copy(),
                                               n_steps, dt, dx, C, KAPPA)

# Measure energies
E_opp_init = np.sum(np.abs(psi_hist_opp[0])**2) * dx
E_opp_final = np.sum(np.abs(psi_hist_opp[-1])**2) * dx
E_same_init = np.sum(np.abs(psi_hist_same[0])**2) * dx
E_same_final = np.sum(np.abs(psi_hist_same[-1])**2) * dx

diff_pct = 100 * (E_same_final - E_opp_final) / (E_opp_final + 1e-30)

print("\n" + "=" * 60)
print("CHARGE FROM PHASE RESULTS")
print("=" * 60)
print(f"  Opposite phases (attract):")
print(f"    Initial energy: {E_opp_init:.4f}")
print(f"    Final energy:   {E_opp_final:.4f}")
print(f"  Same phase (repel):")
print(f"    Initial energy: {E_same_init:.4f}")
print(f"    Final energy:   {E_same_final:.4f}")
print(f"  Energy difference: {diff_pct:+.2f}%")
h0_rejected = abs(diff_pct) > 5.0
print(f"\n  H0 {'REJECTED' if h0_rejected else 'FAILED TO REJECT'}: "
      f"phase {'does' if h0_rejected else 'may not'} determine interaction")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
times_to_plot = [0, 150, 350, n_steps - 1]

# Top-left: opposite phases |Psi|^2
ax = axes[0, 0]
for t in times_to_plot:
    ax.plot(x, np.abs(psi_hist_opp[t])**2, alpha=0.7, label=f't={t}')
ax.set_title('OPPOSITE PHASES (attract): |Psi|^2')
ax.set_xlabel('x')
ax.set_ylabel('|Psi|^2')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# Top-right: chi response opposite
ax = axes[0, 1]
ax.plot(x, chi_hist_opp[0], alpha=0.7, label='t=0')
ax.plot(x, chi_hist_opp[-1], alpha=0.7, label=f't={n_steps-1}')
ax.axhline(CHI_0, color='gray', ls='--', alpha=0.5)
ax.set_title('OPPOSITE PHASES: chi response')
ax.set_xlabel('x')
ax.set_ylabel('chi')
ax.legend()
ax.grid(alpha=0.3)

# Bottom-left: same phase |Psi|^2
ax = axes[1, 0]
for t in times_to_plot:
    ax.plot(x, np.abs(psi_hist_same[t])**2, alpha=0.7, label=f't={t}')
ax.set_title('SAME PHASE (repel): |Psi|^2')
ax.set_xlabel('x')
ax.set_ylabel('|Psi|^2')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# Bottom-right: chi response same
ax = axes[1, 1]
ax.plot(x, chi_hist_same[0], alpha=0.7, label='t=0')
ax.plot(x, chi_hist_same[-1], alpha=0.7, label=f't={n_steps-1}')
ax.axhline(CHI_0, color='gray', ls='--', alpha=0.5)
ax.set_title('SAME PHASE: chi response')
ax.set_xlabel('x')
ax.set_ylabel('chi')
ax.legend()
ax.grid(alpha=0.3)

plt.suptitle('Charge from Phase: Complex Wave Field Interaction',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Key Result

- **Phase** of the complex wave field determines electromagnetic behavior
- Same phase ($\Delta\theta = 0$): constructive interference $\rightarrow$ repulsion (like charges)
- Opposite phase ($\Delta\theta = \pi$): destructive interference $\rightarrow$ attraction (unlike charges)
- **Electric charge is not a separate quantity** \u2014 it\u2019s the phase of $\Psi$